In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
%cd /content/drive/MyDrive/Colab Notebooks/크롤링

/content/drive/MyDrive/Colab Notebooks/크롤링


In [2]:
!pip install requests beautifulsoup4 pandas lxml

In [ ]:
NAVER_SEARCH_CLIENT_ID = ""
NAVER_SEARCH_CLIENT_SECRET = ""

In [11]:
import os
import requests
import pandas as pd
import re
from html import unescape

# 네이버 검색 API 키
# 실제 사용 시에는 코드에 직접 쓰지 말고 .env 또는 Colab 보안 변수 사용 권장


def clean_html(text):
    """
    네이버 API 응답에 포함된 <b> 태그 등 HTML 태그 제거
    """
    text = re.sub(r"<.*?>", "", text)
    return unescape(text).strip()


def make_safe_filename(keyword):
    """
    파일명에 사용할 수 없는 특수문자 제거
    공백은 _로 변경
    """
    filename = keyword.strip()
    filename = re.sub(r"\s+", "_", filename)
    filename = re.sub(r'[\\/:*?"<>|]', "", filename)
    return filename


def search_naver_news(query, display=10, start=1, sort="date"):
    """
    네이버 뉴스 검색 API 호출 함수

    query: 검색 키워드
    display: 한 번에 가져올 뉴스 개수, 최대 100
    start: 검색 시작 위치
    sort: sim = 정확도순, date = 날짜순
    """
    url = "https://openapi.naver.com/v1/search/news.json"

    headers = {
        "X-Naver-Client-Id": NAVER_SEARCH_CLIENT_ID,
        "X-Naver-Client-Secret": NAVER_SEARCH_CLIENT_SECRET
    }

    params = {
        "query": query,
        "display": display,
        "start": start,
        "sort": sort
    }

    response = requests.get(url, headers=headers, params=params)

    print("상태 코드:", response.status_code)

    if response.status_code != 200:
        print("에러 응답:", response.text)
        return []

    data = response.json()
    return data["items"]


# 수집할 키워드 설정
keyword = "창업"

# 뉴스 검색 실행
items = search_naver_news(keyword, display=10, start=1, sort="date")

# 뉴스 데이터를 리스트에 저장
news_list = []

for item in items:
    news_list.append({
        "keyword": keyword,
        "title": clean_html(item["title"]),
        "description": clean_html(item["description"]),
        "originallink": item.get("originallink"),
        "naver_link": item.get("link"),
        "pubDate": item.get("pubDate")
    })

# DataFrame 생성
df = pd.DataFrame(news_list)

# 현재 코드가 실행되는 위치 확인
current_dir = os.getcwd()
print("현재 실행 위치:", current_dir)

save_dir = "/content/drive/MyDrive/Colab Notebooks/크롤링/data"
os.makedirs(save_dir, exist_ok=True)

# 파일명 생성
safe_keyword = make_safe_filename(keyword)
file_name = f"{safe_keyword}1.csv"

# 최종 저장 경로
save_path = os.path.join(save_dir, file_name)

# CSV 저장
df.to_csv(save_path, index=False, encoding="utf-8-sig")

print("CSV 저장 완료:", save_path)

# 결과 확인
df

상태 코드: 200
현재 실행 위치: /content/drive/MyDrive/Colab Notebooks/크롤링
CSV 저장 완료: /content/drive/MyDrive/Colab Notebooks/크롤링/data/창업1.csv


,keyword,title,description,originallink,naver_link,pubDate
0,창업,"경기도, '재도전학교' 수료생 67% 취업 성공",취업과 창업 실패 경험이 있는 도민들에게 심리 회복과 재도전 기회를 제공하는 '경기...,http://www.metroseoul.co.kr/article/2026051450...,http://www.metroseoul.co.kr/article/2026051450...,"Thu, 14 May 2026 15:28:00 +0900"
1,창업,포스코홀딩스-산업은행 맞손…비수도권 벤처기업 육성,한편 포스코그룹은 서울과 포항에 이어 지난해 11월 광양에 벤처 창업보육센터인 '그...,https://www.pointe.co.kr/news/articleView.html...,https://www.pointe.co.kr/news/articleView.html...,"Thu, 14 May 2026 15:28:00 +0900"
2,창업,"하나증권, 위벤처스와 생산적 금융 협력을 위한 업무협약 체결 내용은?",이번 협약을 통해 양사는 미래 신산업 육성과 기술 창업 생태계 스케일업 지원을 위한...,http://www.newsworker.co.kr/news/articleView.h...,http://www.newsworker.co.kr/news/articleView.h...,"Thu, 14 May 2026 15:28:00 +0900"
3,창업,"현대차그룹, 양재사옥에서 '로비 스토리 타운홀' 열어",정의선 회장은 임직원들의 많은 사랑을 받고 있는 식당에 대해 설명하며 정주영 창업회...,http://autotimes.co.kr/detail.php?number=68770...,http://autotimes.co.kr/detail.php?number=68770...,"Thu, 14 May 2026 15:28:00 +0900"
4,창업,“3년 넣었더니 2200만원”...청년미래적금 혜택 얼마나 되길래,"이 위원장은 “청년이 자산을 만들 수 있어야 결혼도, 주거도, 창업도, 도전도 가능...",https://www.ekn.kr/web/view.php?key=2026051402...,https://www.ekn.kr/web/view.php?key=2026051402...,"Thu, 14 May 2026 15:28:00 +0900"
5,창업,"하나증권, 벤처캐피탈 '위벤처스'와 생산적금융 협업",하나증권이 벤처캐피탈 위벤처스와 기술 창업 생태계 스케일업 지원에 나선다. 하나증권...,https://www.newsis.com/view/NISX20260514_00036...,https://n.news.naver.com/mnews/article/003/001...,"Thu, 14 May 2026 15:27:00 +0900"
6,창업,한남대 '2026 창업 페스티벌',한남대가 지역생태계 활성화와 예비창업자 발굴을 위한 '2026 창업 페스티벌'을 개...,https://www.veritas-a.com/news/articleView.htm...,https://www.veritas-a.com/news/articleView.htm...,"Thu, 14 May 2026 15:26:00 +0900"
7,창업,"민주당 김병욱 성남시장 후보, 오리역세권 '제4테크노밸리' 조성 발표","여기에 더해 미래산업 R&D 존에는 팹리스 클러스터와 AI·로봇 자율주행 연구시설,...",https://www.joongdo.co.kr/web/view.php?key=202...,https://www.joongdo.co.kr/web/view.php?key=202...,"Thu, 14 May 2026 15:26:00 +0900"
8,창업,"전북창조경제혁신센터, '2026년 창업-BuS 호남권 연합 IR 개최…지역 투...",전북창조경제혁신센터(대표 박선종)는 14일 전북테크비즈센터에서 전북·전남·광주·제주...,https://www.etnews.com/20260514000417,https://n.news.naver.com/mnews/article/030/000...,"Thu, 14 May 2026 15:26:00 +0900"
9,창업,"최휘영 문체부 장관, 밀양시 문화·관광 현장 찾는다","최 장관은 지역 서점을 필두로, 공연 예술, 지역문화·관광 창업 등 여러 업계 관계...",https://www.dt.co.kr/article/12062493?ref=naver,https://n.news.naver.com/mnews/article/029/000...,"Thu, 14 May 2026 15:24:00 +0900"


In [5]:
pip install newspaper3k

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 28.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.1/211.1 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 8.0 MB/s eta 0:00:00
  Created wheel for tinysegmenter: filename=tinysegmenter-0.3-py3-none-any.whl size=13540 sha256=4c6b43ee102c530562ca8090a05ff7797ec5b4fd51cc33f7723335c5efdfa5c2
  Stored in directory: /root/.cache/pip/wheels/a5/91/9f/00d66475960891a64867914273fcaf78df6cb04d905b104a2a
  Created wheel for feedfinder2: filename=feedfinder2-0.0.4-py3-none-any.whl size=3341 sha256=0136429cdc918f2a31acefa731075244abe0cd88a0d9c1bf408163fc00e292d2
  Stored in directory: /root/.cache/pip/wheels/9f/9f/fb/364871d7426d3cdd4d293dcf7e53d97f16

In [6]:
!pip install newspaper3k tqdm

In [8]:
!pip install lxml_html_clean

In [13]:
import pandas as pd
import os
import requests
from bs4 import BeautifulSoup
from newspaper import Article, Config
from tqdm import tqdm

def fetch_article_content(url):
    """URL에서 본문을 추출하는 함수 (네이버 뉴스 특화 처리 포함)"""
    if pd.isna(url) or not isinstance(url, str) or url.strip() == "":
        return ""

    try:
        # 사람(브라우저)이 접속하는 것처럼 속이기 위한 User-Agent 설정
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
        }

        # 1. 네이버 뉴스인 경우 전용 크롤링 (정확도 100%)
        if "n.news.naver.com" in url:
            res = requests.get(url, headers=headers)
            soup = BeautifulSoup(res.text, 'html.parser')

            # 네이버 뉴스 본문 영역 (id="dic_area") 추출
            article_body = soup.select_one('#dic_area')

            if article_body:
                # 불필요한 태그 지우고 텍스트만 깔끔하게 추출
                return article_body.get_text(separator='\n', strip=True)
            else:
                # 스포츠/연예 뉴스 등 구조가 다른 네이버 뉴스일 경우 범용으로 넘어감
                pass

        # 2. 네이버 뉴스가 아니거나, 위에서 본문을 못 찾은 경우 newspaper3k 사용
        config = Config()
        config.browser_user_agent = headers['User-Agent']
        article = Article(url, language='ko', config=config)
        article.download()
        article.parse()

        content = article.text.strip()

        # 혹시라도 범용 라이브러리가 네이버 안내문구를 또 잡아온다면 차단
        if "기사 섹션 분류 안내" in content and len(content) < 200:
            return ""

        return content

    except Exception as e:
        # 에러 발생 시 진행을 멈추지 않고 빈 값을 반환
        return ""

# 1. 구글 드라이브 기본 경로 설정
base_dir = "/content/drive/MyDrive/Colab Notebooks/크롤링/data"
file_path = os.path.join(base_dir, "창업1.csv")

# 2. CSV 파일 불러오기
print(f"파일 불러오는 중: {file_path}")
if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print(f"✅ 파일을 성공적으로 불러왔습니다. 총 {len(df)}개의 기사가 있습니다.")
else:
    print("❌ 파일을 찾을 수 없습니다. 위 경로에 '창업1.csv' 파일이 있는지 확인해주세요.")
    df = None

if df is not None:
    print("\n본문 수집을 시작합니다. (시간이 조금 걸릴 수 있습니다.)")

    # 3. 반복문을 돌며 본문 수집 (tqdm으로 진행 상태 표시)
    contents = []
    for index, row in tqdm(df.iterrows(), total=len(df)):
        # 네이버 링크가 있으면 네이버 링크를, 없으면 원본 링크를 우선 사용
        target_url = row['naver_link'] if pd.notna(row['naver_link']) else row['originallink']

        # 본문 추출 함수 호출 및 리스트 추가
        content = fetch_article_content(target_url)
        contents.append(content)

    # 4. 데이터프레임에 본문(content) 컬럼 추가
    df['content'] = contents

    # 5. 결과 저장 (새로운 이름으로 저장하여 원본 보호)
    save_path = os.path.join(base_dir, "창업1_본문포함.csv")
    df.to_csv(save_path, index=False, encoding="utf-8-sig")

    print(f"\n🎉 모든 작업이 완료되었습니다! 저장 위치: {save_path}")

# 최종 결과 확인 (데이터프레임 상위 5개 출력)
if df is not None:
    display(df.head())

파일 불러오는 중: /content/drive/MyDrive/Colab Notebooks/크롤링/data/창업1.csv
✅ 파일을 성공적으로 불러왔습니다. 총 10개의 기사가 있습니다.

본문 수집을 시작합니다. (시간이 조금 걸릴 수 있습니다.)


100%|██████████| 10/10 [00:16<00:00,  1.66s/it]


🎉 모든 작업이 완료되었습니다! 저장 위치: /content/drive/MyDrive/Colab Notebooks/크롤링/data/창업1_본문포함.csv


,keyword,title,description,originallink,naver_link,pubDate,content
0,창업,"경기도, '재도전학교' 수료생 67% 취업 성공",취업과 창업 실패 경험이 있는 도민들에게 심리 회복과 재도전 기회를 제공하는 '경기...,http://www.metroseoul.co.kr/article/2026051450...,http://www.metroseoul.co.kr/article/2026051450...,"Thu, 14 May 2026 15:28:00 +0900",경기 재도전학교 ′2026년 제1기 수료식′ 사진 / 경기도 제공\n\n경기도가 취...
1,창업,포스코홀딩스-산업은행 맞손…비수도권 벤처기업 육성,한편 포스코그룹은 서울과 포항에 이어 지난해 11월 광양에 벤처 창업보육센터인 '그...,https://www.pointe.co.kr/news/articleView.html...,https://www.pointe.co.kr/news/articleView.html...,"Thu, 14 May 2026 15:28:00 +0900",포스코홀딩스가 14일 그라운드 광양에서 한국산업은행과 지역 벤처 생태계 활성화를 위...
2,창업,"하나증권, 위벤처스와 생산적 금융 협력을 위한 업무협약 체결 내용은?",이번 협약을 통해 양사는 미래 신산업 육성과 기술 창업 생태계 스케일업 지원을 위한...,http://www.newsworker.co.kr/news/articleView.h...,http://www.newsworker.co.kr/news/articleView.h...,"Thu, 14 May 2026 15:28:00 +0900",정영균 하나증권 IB그룹장(왼쪽)과 하태훈 위벤처스 대표이사가 7일 하나증권에서 열...
3,창업,"현대차그룹, 양재사옥에서 '로비 스토리 타운홀' 열어",정의선 회장은 임직원들의 많은 사랑을 받고 있는 식당에 대해 설명하며 정주영 창업회...,http://autotimes.co.kr/detail.php?number=68770...,http://autotimes.co.kr/detail.php?number=68770...,"Thu, 14 May 2026 15:28:00 +0900",-1년 11개월 리노베이션 거쳐 개장\n\n-새 공간에 깃든 철학과 방향성 공유\n...
4,창업,“3년 넣었더니 2200만원”...청년미래적금 혜택 얼마나 되길래,"이 위원장은 “청년이 자산을 만들 수 있어야 결혼도, 주거도, 창업도, 도전도 가능...",https://www.ekn.kr/web/view.php?key=2026051402...,https://www.ekn.kr/web/view.php?key=2026051402...,"Thu, 14 May 2026 15:28:00 +0900",▲청년미래적금은 만 19~34세 청년을 대상으로 하는 3년 만기 정책형 적금 상품이...
